# SinMorphy Automation V6 (First Output per POS)

Updates: Now correctly loops horizontally through 'POS' and 'Analysis' columns, keeping only the first recommended Lemma and Morphology per unique POS tag.

In [1]:
pip install webdriver_manager


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import time
import traceback
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
def parse_results(page_html):
    """
    V7 Logic:
    1. Extracts the FIRST lemma and morphology for EACH unique POS column.
    2. Mirrors the specific overarching POS tag for EVERY lemma/morphology item so elements map 1-to-1.
    3. Handles 2-column rows (Proper Noun/Invalid) by treating Word as Lemma.
    4. CLEANS invalid lemmas (containing non-Sinhala chars like 'xva') -> Reverts to Word.
    """
    import re
    results = []
    
    # 1. Find table
    table_match = re.search(r'<table.*?>(.*?)</table>', page_html, re.DOTALL | re.IGNORECASE)
    if not table_match:
        return results
        
    table_content = table_match.group(1)
    
    # 2. Split rows
    rows = re.findall(r'<tr.*?>(.*?)</tr>', table_content, re.DOTALL | re.IGNORECASE)
    
    for row in rows:
        if 'Text :' in row:
            continue
            
        cells = re.findall(r'<td.*?>(.*?)</td>', row, re.DOTALL | re.IGNORECASE)
        
        # Skip empty/broken rows
        if not cells or len(cells) < 2:
            continue
            
        # Cell 0: Word
        word = re.sub(r'<.*?>', '', cells[0]).strip()
        
        all_lemmas = []
        all_morphologies = []
        all_tags = []
        
        # Base POS will just be the very first POS encountered
        pos = "Proper noun or Invalid" 
        
        if len(cells) >= 3:
            # We iterate through cells in pairs: (POS, Analysis)
            pos_index = 1
            analysis_index = 2
            
            while analysis_index < len(cells):
                cur_pos = re.sub(r'<.*?>', '', cells[pos_index]).strip()
                if pos == "Proper noun or Invalid": # Set base POS to first valid
                    pos = cur_pos
                    
                analysis_raw = cells[analysis_index]
                analyses = re.split(r'<br\s*/?>', analysis_raw, flags=re.IGNORECASE)
                analyses = [line for line in analyses if re.sub(r'<.*?>', '', line).strip()]
                
                # Check if we already have this POS tag recorded (to only keep the FIRST of each POS)
                if cur_pos not in all_tags and analyses:
                    first_line = analyses[0] # Only take the first morphology line
                    clean_line = re.sub(r'<.*?>', '', first_line).strip()
                    
                    if clean_line:
                        parts = clean_line.split()
                        
                        if len(parts) >= 2:
                            analysis_str = parts[-1] 
                        elif len(parts) == 1:
                            analysis_str = parts[0]
                        else:
                            analysis_str = ""
                            
                        if analysis_str:
                            parts_split = analysis_str.split('+')
                            cur_lemma = parts_split[0] if parts_split else ""
                            cur_morph = analysis_str
                            
                            all_lemmas.append(cur_lemma)
                            all_morphologies.append(cur_morph)
                            all_tags.append(cur_pos)
                            
                pos_index += 2
                analysis_index += 2
                
            if all_lemmas:
                lemma = all_lemmas[0]
                morphology = all_morphologies[0]
            else:
                lemma = ""
                morphology = ""
                
            if not all_tags:
                all_tags = [""]
        else:
            # Edge Case: 2 Columns (Proper noun or Invalid)
            pos = re.sub(r'<.*?>', '', cells[1]).strip() if len(cells) > 1 else pos
            lemma = word
            morphology = "NA"
            all_lemmas = [lemma]
            all_morphologies = [morphology]
            all_tags = [pos]
        
        # -----------------------------------------------------------------
        # CLEANUP LOGIC (Fix xva/non-Sinhala lemmas)
        # -----------------------------------------------------------------
        if not lemma or lemma == 'None':
            lemma = word
        elif re.search(r'[^඀-෿].*', lemma):
            lemma = word
            
        cleaned_all_lemmas = []
        for l in all_lemmas:
            if not l or l == 'None':
                cleaned_all_lemmas.append(word)
            elif re.search(r'[^඀-෿].*', l):
                cleaned_all_lemmas.append(word)
            else:
                cleaned_all_lemmas.append(l)
        all_lemmas = cleaned_all_lemmas
            
        # -----------------------------------------------------------------

        results.append({
            'word': word,
            'pos': pos,
            'lemma': lemma,
            'morphology': morphology,
            'all_tags': all_tags,
            'all_lemmas': all_lemmas,
            'all_morphologies': all_morphologies
        })
            
    return results

In [4]:
def get_lemma_and_pos(current_driver, text, service, chrome_options):
    """
    Returns (driver, page_source) so the main loop can keep track of the driver
    even if we have to restart it due to a crash.
    """
    MAX_RETRIES = 3
    URL = 'http://nlp-tools.uom.lk/sin-morphy/'
    
    import time
    import traceback
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    
    driver = current_driver
    
    for attempt in range(MAX_RETRIES):
        try:
            # 1. Navigate
            driver.get(URL)
            # Small sleep to let JS init
            time.sleep(1)
            
            # 2. Locate
            text_area = WebDriverWait(driver, 15).until(
                EC.visibility_of_element_located((By.ID, 'anaText'))
            )
            
            # 3. Interact
            text_area.clear()
            text_area.send_keys(text)
            
            # 4. Click button
            analyze_btn = driver.find_element(By.NAME, 'analyseButton')
            analyze_btn.click()
            
            # 5. Wait for results
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, 'table'))
            )
            
            return driver, driver.page_source
            
        except Exception as e:
            error_detail = repr(e)
            print(f"Attempt {attempt + 1}/{MAX_RETRIES} failed. Error: {error_detail}")
            time.sleep(2)
            
            # CRITICAL FIX: If the driver has hung (like a urllib3 ReadTimeoutError),
            # we must forcefully terminate the dead driver and create a new one.
            try:
                driver.quit()
            except:
                pass
            print("Restarting webdriver...")
            driver = webdriver.Chrome(service=service, options=chrome_options)
             
    print("All retries failed.")
    return driver, None

In [5]:
# Configuration
INPUT_FILE = 'data/century_tokens.json'
OUTPUT_FILE = 'data/century_analysis_results_v6.json'
CHECKPOINT_INTERVAL = 10

# Load Data
import os
import sys

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        century_data = json.load(f)
else:
    print(f"File not found: {INPUT_FILE}")
    century_data = {}

# Initialize Driver
chrome_options = Options()
# chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

final_results = {}
# Load existing if any
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        final_results = json.load(f)
    print(f"Resumed from {sum([len(x) for x in final_results.values()])} total sentences.")

try:
    for century_key, sentences in tqdm(century_data.items(), desc="Processing Keys"):
        if century_key not in final_results:
            final_results[century_key] = []
        
        existing_count = len(final_results[century_key])
        
        for i, tokens in enumerate(sentences):
            if i < existing_count:
                continue
                
            sentence_text = ' '.join(tokens)
            
            # Get HTML with robust restart-aware retry logic
            driver, page_source = get_lemma_and_pos(driver, sentence_text, service, chrome_options)
            
            # GUARANTEE NO SKIPS: Stop completely if we can't get the page source
            # By raising an error here, the sentence is NEVER appended to final_results
            # Thus, when you re-run the notebook, it resumes exactly from this failed sentence.
            if not page_source:
                print(f"\nFATAL ERROR: Failed exactly at century {century_key}, sentence index {i}.")
                print("Stopping execution to prevent data loss. You can just re-run the cell to resume from here.")
                raise RuntimeError("Failed to extract data from SinMorphy. Halting so no words are skipped.")
                
            # Process if successful
            analysis = parse_results(page_source)
            
            final_results[century_key].append({
                'sentence_index': i,
                'original_text': sentence_text,
                'analysis': analysis
            })
            
            # Optional: Periodic driver restart to prevent massive memory leaks
            if (i + 1) % 500 == 0:
                print(f"Periodic driver restart at iteration {i} to free memory...")
                try: driver.quit()
                except: pass
                driver = webdriver.Chrome(service=service, options=chrome_options)
            
            if i % CHECKPOINT_INTERVAL == 0:
                with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                    json.dump(final_results, f, ensure_ascii=False, indent=2)
                    
except KeyboardInterrupt:
    print("Stopped by user.")
except Exception as e:
    print(f"Exception: {e}")
finally:
    try: driver.quit()
    except: pass
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(final_results, f, ensure_ascii=False, indent=2)
    print("Saved checkpoint.")

Processing Keys:  22%|██▏       | 2/9 [05:35<20:45, 177.91s/it]

Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Periodic driver restart at iteration 499 to free memory...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: TimeoutException()
Restarting webdriver...


Processing Keys:  44%|████▍     | 4/9 [1:41:32<2:25:47, 1749.55s/it]

Periodic driver restart at iteration 499 to free memory...


Processing Keys:  78%|███████▊  | 7/9 [2:33:53<39:24, 1182.23s/it]  

Periodic driver restart at iteration 499 to free memory...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...
Attempt 1/3 failed. Error: ReadTimeoutError("HTTPConnectionPool(host='localhost', port=51940): Read timed out. (read timeout=120)")
Restarting webdriver...


Processing Keys:  89%|████████▉ | 8/9 [3:31:13<31:40, 1900.80s/it]

Periodic driver restart at iteration 499 to free memory...
Periodic driver restart at iteration 999 to free memory...
Periodic driver restart at iteration 1499 to free memory...


Processing Keys: 100%|██████████| 9/9 [4:46:47<00:00, 1911.97s/it]


Saved checkpoint.


In [6]:
def extract_morphology_data(input_dict):
    rows = []
    
    for century, sentences in input_dict.items():
        if not isinstance(sentences, list):
            continue
            
        for sentence in sentences:
            s_idx = sentence.get("sentence_index")
            analysis_list = sentence.get("analysis") or []
            
            for item in analysis_list:
                # 1. Determine Lemma Strictly from 'lemma' field
                new_lemma = item.get("lemma")
                word = item.get("word", "")
                
                # Fallback to word if lemma is missing
                if not new_lemma or new_lemma == "None":
                    new_lemma = word

                # 2. Report other fields (Morphology not considered for Lemma)
                full_morph = item.get("morphology", "") or ""
                pos = item.get("pos", "")
                
                # Extract all possible tags, lemmas and morphologies
                all_tags = item.get("all_tags", [])
                all_lemmas = item.get("all_lemmas", [])
                all_morphologies = item.get("all_morphologies", [])
                
                rows.append({
                    "century": century,
                    "sentence_index": s_idx,
                    "word": word,
                    "new_lemma": new_lemma,
                    "morphology": full_morph,
                    "pos": pos,
                    "all_tags": " | ".join(all_tags) if isinstance(all_tags, list) else all_tags,
                    "all_lemmas": " | ".join(all_lemmas) if isinstance(all_lemmas, list) else all_lemmas,
                    "all_morphologies": " | ".join(all_morphologies) if isinstance(all_morphologies, list) else all_morphologies
                })
                
    return pd.DataFrame(rows)

In [7]:
df_final = extract_morphology_data(final_results)
print(f'Total rows: {len(df_final)}')
df_final.to_csv("data/century_morphology_data_v6.csv", index=False, encoding='utf-8-sig')
df_final.head(50)

Total rows: 64813


,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC
5,5,0,යන්,යන්,+N+F+PL+ACC,NOUN,NOUN,යන්,+N+F+PL+ACC
6,5,0,ලොකත්රයට,ලොකත්රයට,NA,Proper noun or Invalid,Proper noun or Invalid,ලොකත්රයට,NA
7,5,0,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ+GUESS+N+RT,NOUN,NOUN,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ+GUESS+N+RT
8,5,0,වූ,ව,ව+V+NFIN+PASP,Verb,Verb,ව,ව+V+NFIN+PASP
9,5,0,මුනීන්ද්ර,මුනීන්ද්ර,NA,Proper noun or Invalid,Proper noun or Invalid,මුනීන්ද්ර,NA


In [8]:
print(len(df_final[df_final['century'] == '5']))
print(len(df_final[df_final['century'] == '12']))
print(len(df_final[df_final['century'] == '13']))
print(len(df_final[df_final['century'] == '14'])) 
print(len(df_final[df_final['century'] == '15']))
print(len(df_final[df_final['century'] == '16']))
print(len(df_final[df_final['century'] == '18']))
print(len(df_final[df_final['century'] == '19']))
print(len(df_final[df_final['century'] == '20']))

565
866
13617
5210
8483
1369
4707
12701
17295


In [9]:
df_final.isna().sum()

century             0
sentence_index      0
word                0
new_lemma           0
morphology          0
pos                 0
all_tags            0
all_lemmas          0
all_morphologies    0
dtype: int64

In [10]:
import re

In [11]:
def is_sinhala(text):
    # This pattern matches any character in the Sinhala Unicode range
    non_sinhala_pattern = re.compile(r'[^\u0D80-\u0DFF\s\d.,?!\-()\'"]')
    return bool(non_sinhala_pattern.search(text))

In [12]:
def get_non_sinhala_part(text):
    if not isinstance(text, str):
        return ""
    
    match = re.search(r'[^\u0D80-\u0DFF].*', text)
    
    return match.group(0) if match else ""

In [13]:
df_final[['word', 'new_lemma']][df_final['new_lemma'].apply(is_sinhala)]

,word,new_lemma


In [14]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

In [15]:
# df_final['new_lemma']

In [16]:
# df_final['non_sinhala_parts'] = df_final['new_lemma'].apply(get_non_sinhala_part)

In [17]:
# df_final[df_final['non_sinhala_parts'] == 'xThe']

In [18]:
# final_results['13']

In [19]:
# def fix_dictionary(data):
#     count = 0
#     total_processed = 0
    
#     for century, items in data.items():
#         for item in items:
#              analysis_list = item.get('analysis', [])
             
#              for entry in analysis_list:
#                  # Check 'lemma' field
#                  lemma = entry.get('lemma')
#                  word = entry.get('word')
                 
#                  if not lemma: 
#                      continue
                 
#                  non_sinhala_pattern = re.compile(r'[^\u0D80-\u0DFF\s\d.,?!\-()\'"]')
#                  bool(non_sinhala_pattern.search(text))

#                  if bool(non_sinhala_pattern.search(lemma)):
#                      entry['lemma'] = word
#                      count += 1
                 
#                  total_processed += 1
                 
#     print(f"Scanning complete. Fixed {count} suspect lemmas.")
#     return data

In [20]:
# final_results_1 = fix_dictionary(final_results)

In [21]:
# df_final_1 = extract_morphology_data(final_results_1)
# print(f'Total rows: {len(df_final_1)}')
# df_final_1.head(40)

In [22]:
# df_final_1[['word', 'new_lemma']][df_final_1['new_lemma'].apply(is_sinhala)]